# TP : Spark Streaming

Le but de ce TP est d'explorer la librairie de calcul Spark Streaming, et de mettre en évidence les particularités de traitement de flux par rapport aux données statiques.

# Cas d'usage

Nous sommes toujours sur notre produit PomSort. On s'intéresse plus particulièrement au reporting en temps réel : nous allons mettre en place les traitements qui pourraient alimenter ce reporting.

# Données

Les données sont fournies sous forme de fichiers CSV dans le répertoire `data/`. Bien que ce soit des fichiers déjà constitués, nous demanderons à Spark Streaming de les traiter comme des flux, en simulant l'écoulement du temps grâce à une colonne `timestamp`.

Dans cette introduction, on affiche le début des fichiers avec Pandas, pour montrer leur structure.

## Poids des pommes

La balance pèse en continu ce qui passe sur le tapis. Toutes les secondes exactement, on a une mesure : `weight`, exprimée en grammes.

In [ ]:
import pandas as pd
pd.read_csv('data/weights/weights.csv').head()

## Diamètres des pommes

Le diamètre est mesuré par un algorithme à chaque fois qu'une pomme est détectée sur le tapis d'arrivée. Les timestamps sont donc irréguliers. On a une mesure à chaque fois : `diameter`, exprimée en centimètres.

In [ ]:
pd.read_csv('data/diameters/diameters.csv').head()

## Identification des variétés de pommes

L'identification se fait à la détection des pommes, comme pour la mesure du diamètre. Cependant, le modèle de classification étant plus lent que celui de mesure du diamètre, le timestamp est décalé de quelques secondes par rapport à celui-ci.

On a une colonne supplémentaire, `identification_id`, unique pour chaque événement d'identification.

In [ ]:
pd.read_csv('data/identifications/identifications.csv').head()

# Initialisation de Spark

In [ ]:
%pip install pyspark

⚠️ **N'exécuter la cellule ci-dessous que si vous êtes sous Windows**⚠️

In [ ]:
import os

HADOOP_HOME = r'Y:\BUT3 VCOD\BigData'

os.environ['HADOOP_HOME'] = HADOOP_HOME
os.environ['PATH'] += os.pathsep + os.path.join(HADOOP_HOME, 'bin')

Création de la session Spark :

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("PomSortStreaming") \
    .getOrCreate()

On charge aussi la librairie des fonctions PySpark :

In [ ]:
from pyspark.sql import functions as F

# Tutoriel : Création d'un dataframe de streaming pour les poids

Prenez le temps d'exécuter ce tutoriel en en comprenant la logique.

Pour créer un dataframe de streaming, il faut quelques ingrédients :
- le **schéma** : structure (colonnes et types) des données élémentaires qui sont dans le flux
- le **format** de représentation : ici, CSV
- l'**emplacement** : c'est forcément un répertoire. Chaque fichier est donc dans son propre répertoire, pour que Spark Streaming ne les mélange pas

In [ ]:
# Le schéma est donné sous forme de chaîne analogue à la définition de colonnes en SQL
weights_schema = 'timestamp timestamp, weight float'

# Le dataframe lui-même
weights = spark.readStream.schema(weights_schema).format('csv').option('header', True).load('data/weights')

Le résultat est un dataframe Spark, qui a la particularité d'être "streamable".

In [ ]:
type(weights)

In [ ]:
weights.isStreaming

Pour afficher le contenu du dataframe, il faut créer une requête. En streaming, le dataframe ne mémorise pas les données : ce n'est qu'un tuyau branché sur la source, et c'est la requête qui "aspire" les données. Ici elle sort les données sur la console = la sortie de la cellule Jupyter.

Comme on est en streaming, les données sont infinies et la requête ne s'arrête jamais : on lui demande de s'interrompre toute seule au bout de quelques secondes, ce qui est suffisant pour notre volume. Par défaut, sur la console elle n'affiche que 20 lignes de résultat.

In [ ]:
def show_result(df, timeout=30, complete=False):
    writer = df.writeStream
    if complete:
        writer = writer.outputMode('complete')
    query = writer.format("console").start()
    query.awaitTermination(timeout=timeout)

show_result(weights)

## Opérations de base sur les dataframes

Ces opérations seront utiles dans la suite. Elles illustrent les indications du cours.

NB : elles ne sont pas propres au streaming. A chaque fois, un nouveau dataframe est créé à partir de celui qui est utilisé dans l'instruction (pour rappel, en streaming les dataframes ne sont pas stockés en mémoire mais décrivent des transformations)

In [ ]:
# Ajouter une colonne calculée à partir d'une autre
weights_squared = weights.withColumn('weight_kg^2', (F.col('weight') / 1000)**2)
show_result(weights_squared)

In [ ]:
# Renommage d'une colonne
weights_renamed = weights.withColumnRenamed('weight', 'weight_g')
show_result(weights_renamed)

In [ ]:
# Filtre sur les données
weights_heavy = weights.filter(F.col('weight') > 250)
show_result(weights_heavy)

In [ ]:
# Regroupement et calcul d'une statistique
weights_mean_per_minute = weights.withColumn('minute', F.minute(F.col('timestamp'))).groupBy('minute').agg(F.mean('weight'))

# Comme il y a un groupBy, la requête doit attendre l'arrivée des données complètes avant de sortir un résultat fiable
show_result(weights_mean_per_minute, complete=True)

# Exercices

## Création des autres dataframes

En s'inspirant de l'exemple pour les poids, créer un dataframe pour les 2 autres sources : `diameters`, `identifications`.

Attention à bien spécifier le schéma ; pour cela se référer aux fichiers ou aux extraits Pandas en haut du notebook. Voici les types de données utiles pour le schéma :
- pour un timestamp : `timestamp`
- pour un nombre flottant : `float`
- pour un nombre entier : `int`
- pour une chaîne de caractères : `string`

In [ ]:
diameters = ...

show_results(diameters)

In [ ]:
identifications = ...

show_results(identifications)

## Calcul d'une moyenne glissante

Le flux étant infini, une moyenne globale comme en Pandas n'a pas de sens. Il faut forcément la calculer sur des fenêtres temporelles.

En Spark Streaming, cela se fait au moyen d'un regroupement (`groupBy`) sur la fenêtre temporelle. Une fenêtre glissante est construite comme suit :

```
F.window('nom_de_la_colonne_timestamp', 'durée_de_la_fenêtre', 'durée_de_glissement')
```

Sortir la moyenne glissante des diamètres, avec une fenêtre de 1 min qui se décale de 30 s à chaque fois.

In [ ]:
diameter_moving_avg = ...

# Comme il y a un groupBy, la requête doit attendre l'arrivée des données complètes avant de sortir un résultat fiable
show_result(diameter_moving_avg, complete=True)

## Jointure simple entre 2 dataframes

La jointure classique sur des champs fonctionne de la même manière qu'en Spark classique.

Créer un dataframe qui associe, pour chaque diamètre mesuré, la mesure de poids qui a eu lieu au même instant.

In [ ]:
diameters_with_weights = ...

show_result(diameters_with_weights)

## Jointure sur des fenêtres

On va faire une jointure similaire, mais sur le dataframe d'identification des variétés (`identifications`). Pour rappel, les timestamps ne sont pas les mêmes que ceux de la mesure du diamètre : il faut trouver un moyen d'associer les timestamps "proches".

La stratégie retenue est la suivante :
- affectation des mesures de diamètre à une fenêtre temporelle assez grande
- affectation des événements d'identification à une fenêtre identique
- jointure sur la fenêtre
  - chaque fenêtre du résultat "contiendra" _n_ mesures de diamètre et _n'_ événements d'identification
- pour chaque timestamp mesure de diamètre, on cherche l'événement d'identification postérieur le plus proche

### Affectation aux fenêtres

Créer un dataframe dérivé de `diameters_with_weights`, qui contient une colonne supplémentaire : une fenêtre temporelle d'1 minute. La fenêtre doit être non glissante, pour que chaque mesure soit associée à une et une seule fenêtre.

Pour créer une fenêtre non glissante : `F.window('nom_de_la_colonne_timestamp', 'durée_de_la_fenêtre')`

In [ ]:
diameters_with_weights_1m = ...

show_result(diameters_with_weights_1m)

Faire de même à partir du dataframe `identifications`

In [ ]:
identifications_1m = ...

show_result(identifications_1m)

### Jointure sur la fenêtre

Nos 2 dataframes dérivés ont maintenant une fenêtre temporelle commune, sur laquelle on peut les joindre. Faire cette jointure, et filtrer le résultat pour ne garder que les lignes dont `identification_timestamp` est postérieur ou égal à `timestamp`.

Pour filtrer, utiliser la méthode `filter()` d'un dataframe.

Lors de la référence à `identifications_1m`, renommer sa colonne `timestamp` en `identification_timestamp`, pour éviter les ambiguïtés avec le `timestamp` de `diameters_with_weights_1m`. Le renommage se fait avec la fonction `withColumnRenamed()` appliquée à un dataframe.

In [ ]:
diameters_with_weights_and_identifications = ...

show_result(diameters_with_weights_and_identifications)

### Raccrochage de l'événement postérieur le plus proche

Cette étape est un peu plus compliquée. On va regrouper les données de `diameters_with_weights_and_identifications` par `timestamp` (timestamp de mesure du diamètre), et appliquer une fonction Python à tout le groupe pour trouver le timestamp `identification_timestamp` postérieur le plus proche.

Cette fonction Python doit prendre un dataframe Pandas en entrée, en retourner un nouveau. PySpark va mettre bout à bout tous les dataframes Pandas résultant pour chaque valeur de `timestamp`.

Pour ce résultat, les colonnes doivent être :
- `timestamp` (celui du regroupement)
- `diameter` (valeur du diamètre)
- `identification_id` (ID de l'événement postérieur le plus proche)
- `variety` (variété de pomme associée)
- 
Il faudra aussi donner à PySpark le schéma du dataframe ainsi retourné.

In [ ]:
def process_group(df_pandas):
    # Se rappeler qu'ici, on traite du dataframe *Pandas* et pas Spark
    return ...

diameters_with_weights_and_single_identification = ...

show_result(diameters_with_weights_and_single_identification, timeout=60)

## Jointure avec un dataframe statique

Il y a un fichier en plus, dans le fichier `data/truth/truth.csv` : les "vraies" variétés identifiées après annotation par un expert :

In [ ]:
pd.read_csv('data/truth/truth.csv').head()

Lire le fichier sous forme de dataframe statique (non streaming) :

In [ ]:
truth = ...

truth.show()

In [ ]:
truth.isStreaming

Nous pouvons maintenant faire une jointure entre notre dataframe de streaming et celui-ci, pour avoir en même temps l'identification algorithmique et la vraie valeur.

In [ ]:
final_df = ...

show_result(final_df, timeout=60)

## Récupération explicite des données par itération

In [ ]:
final_df.writeStream.foreach(print).start().awaitTermination(timeout=120)